# 00 — Preparação dos dados e divisão por sinalizante

**Trabalho final — Tópicos Especiais em ESSC II (IA aplicada a Séries Temporais), PPgCC UERN/UFERSA**
**Problema:** reconhecer qual de 20 palavras de Libras foi sinalizada, a partir da série temporal de keypoints do corpo e das mãos extraídos de vídeo pelo MediaPipe Holistic.

Este notebook prepara a base usada por **todos** os outros notebooks:

1. carrega as séries do MINDS-Libras (1.158 vídeos, 12 sinalizantes, 20 palavras);
2. identifica sinalizante e repetição de cada vídeo;
3. define a **divisão em 4 grupos de sinalizantes** (o "corte temporal" do nosso problema; ver seção 3);
4. salva um único arquivo leve, `dados/minds_st.npz`, que vai para o Google Drive e é lido no Colab.

Nada aqui é ajustado aos dados de teste: a única transformação feita é a seleção de colunas. A padronização (z-score) é ajustada **dentro de cada rodada, só no treino**, nos notebooks de modelagem.

## 1. De onde vêm as séries (etapas executadas antes deste notebook)

As etapas abaixo rodaram no notebook local (`D:\ProjetoMestrado\notebooks\treino_minds.ipynb`), porque precisam dos 65 GB de vídeos:

| Etapa | O que faz | Script |
|---|---|---|
| 1. Download | 12 zips do MINDS-Libras no Zenodo (registro 2667329), um por sinalizante | `minds_baixar.py` |
| 2. Keypoints | Cada vídeo (1920×1080, ~30 fps, 3 a 7 s) é amostrado em **32 frames uniformes**; o MediaPipe Holistic extrai 33 pontos do corpo, 21 de cada mão e 468 do rosto por frame | `minds_extrair.py` |
| 3. Série temporal | Interpolação de falhas curtas da mão (≤ 4 frames), corte do repouso no início e no fim, reamostragem do trecho ativo de volta para **32 instantes**, normalização pelo corpo (centro nos ombros, escala pela largura dos ombros), mão dominante sempre na mesma coluna (espelhamento) e forma de cada mão relativa ao próprio pulso | `montar_dataset.py` |

Para o estudo de ablação, a etapa 3 foi gerada **sem a média móvel** que o pipeline original aplicava (`--k_suavizar 1`). Assim a base de comparação é o sinal bruto do MediaPipe, e a suavização vira uma das técnicas testadas (notebook 04):

```
python montar_dataset.py --raiz D:\ProjetoMestrado\data\minds_malta --so_familia MINDS_ --k_suavizar 1 --saida <trabalho>\dados\minds_bruto_k1.npz
```

Todas essas transformações são **por vídeo** (usam só os frames do próprio vídeo), então não misturam informação entre treino e teste.

## 2. Configuração

No **Colab**: copie a pasta `trabalho_series_temporais` (com `dados/minds_bruto_k1.npz`) para `Meu Drive` e rode. **Localmente**: abra o notebook de dentro da pasta `notebooks/`.

In [1]:
import sys, re
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/trabalho_series_temporais")
else:
    BASE = Path.cwd().parent if (Path.cwd().parent / "dados").exists() else Path.cwd()
DADOS, FIG, RES = BASE / "dados", BASE / "figuras", BASE / "resultados"
for p in (FIG, RES):
    p.mkdir(exist_ok=True)
print("pasta do trabalho:", BASE)

pasta do trabalho: /home/claude/ts


## 3. Carregar as séries

O arquivo tem três versões de atributos (A, B e C). Usamos a **versão B (321 atributos por instante)**, a melhor nos experimentos anteriores do projeto:

| Bloco | Pontos | Atributos | O que representa |
|---|---|---|---|
| `pose` | 25 pontos do corpo (sem pernas) × (x, y, z) | 75 | posição dos braços, ombros e cabeça |
| `dom` | 21 pontos da mão dominante × (x, y, z) | 63 | posição da mão dominante no espaço do corpo |
| `outra` | 21 pontos da outra mão × (x, y, z) | 63 | posição da outra mão |
| `fdom` | 20 pontos da mão dominante relativos ao pulso | 60 | **formato** da mão dominante (configuração de mão) |
| `foutra` | 20 pontos da outra mão relativos ao pulso | 60 | formato da outra mão |

In [2]:
d = np.load(DADOS / "minds_bruto_k1.npz", allow_pickle=False)
idx = pd.read_csv(DADOS / "minds_bruto_k1_index.csv", encoding="utf-8-sig")
X = d["X_B"].astype(np.float32)          # (N, T, F)
cols = d["cols_B"].astype(str)
mask = d["mask"]                          # (N, T, 4): pose, mão dominante, outra mão, rosto detectados
y = d["y"].astype(np.int64)
classes = d["classes"].astype(str)
arquivo = d["arquivo"].astype(str)
assert (idx.arquivo.to_numpy() == arquivo).all()

N, T, F = X.shape
print(f"séries: {N} | instantes por série: {T} | atributos por instante: {F} | classes: {len(classes)}")
print("NaN:", int(np.isnan(X).sum()), "| faixa:", round(float(X.min()), 2), "a", round(float(X.max()), 2))
bloco = pd.Series(cols).str.extract(r"^([a-z]+)")[0]
print(bloco.value_counts().reindex(["pose", "dom", "outra", "fdom", "foutra"]).to_dict())

séries: 1158 | instantes por série: 32 | atributos por instante: 321 | classes: 20
NaN: 0 | faixa: -7.01 a 6.34
{'pose': 75, 'dom': 63, 'outra': 63, 'fdom': 60, 'foutra': 60}


## 4. Sinalizante e repetição de cada vídeo

O nome de cada arquivo traz a palavra, o sinalizante e a repetição, por exemplo `01AcontecerSinalizador03-4.mp4` (palavra 01, sinalizante 3, 4ª repetição). Cada sinalizante gravou cada palavra até 5 vezes.

In [3]:
sinalizante = np.array([int(re.search(r"Sinalizador(\d+)", a).group(1)) for a in arquivo])
repeticao = np.array([int(re.search(r"-(\d+)\.\w+$", a).group(1)) for a in arquivo])

tab = pd.crosstab(pd.Series(classes[y], name="palavra"), pd.Series(sinalizante, name="sinalizante"))
tab.loc["TOTAL"] = tab.sum()
display(tab)
print("repetições encontradas:", sorted(int(r) for r in set(repeticao)), "(há sinalizantes com 6 ou 7 repetições de uma palavra)")
faltam = (tab.drop("TOTAL") < 5).sum().sum()
print(f"combinações palavra × sinalizante com menos de 5 repetições: {faltam} de {20 * 12}")
print("vídeos por sinalizante:", {int(k): int(v) for k, v in zip(*np.unique(sinalizante, return_counts=True))})

sinalizante,1,2,3,4,5,6,7,8,9,10,11,12
palavra,,,,,,,,,,,,
acontecer,5,5,5,5,5,5,5,5,5,5,5,5
aluno,5,5,0,5,5,5,5,5,5,5,5,5
amarelo,5,5,5,5,5,5,5,5,0,5,5,5
america,5,5,0,5,5,5,5,5,5,5,5,5
aproveitar,5,5,5,5,5,5,5,5,5,5,5,5
bala,5,5,5,5,5,5,5,5,5,5,5,5
banco,5,5,5,5,5,5,5,5,5,5,5,5
banheiro,5,5,5,5,5,5,5,5,0,5,5,5
barulho,5,5,5,5,5,5,7,5,5,5,5,5


repetições encontradas: [1, 2, 3, 4, 5, 6, 7] (há sinalizantes com 6 ou 7 repetições de uma palavra)
combinações palavra × sinalizante com menos de 5 repetições: 9 de 240
vídeos por sinalizante: {1: 100, 2: 100, 3: 85, 4: 95, 5: 100, 6: 100, 7: 103, 8: 100, 9: 75, 10: 100, 11: 100, 12: 100}


## 5. Divisão dos dados: 4 grupos de sinalizantes

**Por que não uma divisão cronológica?** O enunciado proíbe o *K-fold* aleatório porque, numa série única, sortear pontos deixa o modelo ver o futuro. No nosso problema cada vídeo é **uma série independente**, e o vazamento equivalente é outro: o **mesmo sinalizante** aparecer no treino e no teste. Cada pessoa tem um jeito próprio de sinalizar (velocidade, amplitude, proporções do corpo). Se o modelo vê outra repetição da mesma pessoa no treino, ele reconhece a pessoa, e não o sinal.

Por isso a divisão é **por sinalizante**: o teste é sempre feito com pessoas que o modelo nunca viu. Esse é o cenário real de uso (um aplicativo vai ser usado por pessoas que não estavam no treino).

Os 12 sinalizantes foram distribuídos em 4 grupos de 3, intercalados para equilibrar o número de vídeos. Em cada uma das 4 rodadas:

- **teste:** 1 grupo (3 sinalizantes);
- **validação:** 2 sinalizantes do grupo seguinte (para *early stopping* e escolha de hiperparâmetros);
- **treino:** os 7 sinalizantes restantes.

Cada vídeo é testado exatamente uma vez. Nenhum sorteio é envolvido, então a divisão é idêntica em qualquer máquina.

In [4]:
GRUPOS = {0: [1, 5, 9], 1: [2, 6, 10], 2: [3, 7, 11], 3: [4, 8, 12]}
VALIDACAO = {k: GRUPOS[(k + 1) % 4][:2] for k in GRUPOS}      # 2 sinalizantes do grupo seguinte
grupo = np.array([next(g for g, s in GRUPOS.items() if sv in s) for sv in sinalizante])

def particao(rodada):
    # devolve os índices de treino, validação e teste da rodada (0 a 3)
    te = np.where(grupo == rodada)[0]
    va = np.where(np.isin(sinalizante, VALIDACAO[rodada]))[0]
    tr = np.setdiff1d(np.arange(N), np.concatenate([te, va]))
    return tr, va, te

linhas = []
for k in GRUPOS:
    tr, va, te = particao(k)
    s_tr, s_va, s_te = (sorted(int(v) for v in set(sinalizante[i])) for i in (tr, va, te))
    assert not (set(s_tr) & set(s_te)) and not (set(s_va) & set(s_te)) and not (set(s_tr) & set(s_va))
    assert all(len(set(y[i])) == 20 for i in (tr, va, te)), "toda partição precisa ter as 20 palavras"
    linhas.append(dict(rodada=k, treino_sinalizantes=s_tr, validacao=s_va, teste=s_te,
                       n_treino=len(tr), n_validacao=len(va), n_teste=len(te)))
divisao = pd.DataFrame(linhas)
display(divisao)
print("cada vídeo é teste exatamente uma vez:", np.bincount(grupo).sum() == N)

,rodada,treino_sinalizantes,validacao,teste,n_treino,n_validacao,n_teste
0,0,"[3, 4, 7, 8, 10, 11, 12]","[2, 6]","[1, 5, 9]",683,200,275
1,1,"[1, 4, 5, 8, 9, 11, 12]","[3, 7]","[2, 6, 10]",670,188,300
2,2,"[1, 2, 5, 6, 9, 10, 12]","[4, 8]","[3, 7, 11]",675,195,288
3,3,"[2, 3, 6, 7, 9, 10, 11]","[1, 5]","[4, 8, 12]",663,200,295


cada vídeo é teste exatamente uma vez: True


### Protocolo anti-vazamento usado nos próximos notebooks

1. A divisão acima é a única usada; nenhum notebook sorteia vídeos entre treino e teste.
2. A padronização (z-score por atributo) é **calculada só no treino** de cada rodada e aplicada à validação e ao teste (função `padronizar` abaixo, que todos os notebooks repetem).
3. A validação serve para *early stopping* e escolha de hiperparâmetros; o teste só é usado uma vez, no fim de cada rodada.
4. As transformações de pré-processamento da ablação (média móvel, FFT) são feitas **dentro de cada série**, sem parâmetros aprendidos de outras séries.

In [5]:
def padronizar(X, tr):
    # z-score por atributo com média e desvio do TREINO (todas as séries e instantes do treino)
    mu = X[tr].reshape(-1, X.shape[-1]).mean(0)
    sd = X[tr].reshape(-1, X.shape[-1]).std(0) + 1e-6
    return (X - mu) / sd, mu, sd

# demonstração na rodada 0: o treino fica com média 0 e desvio 1; o teste, não exatamente (como deve ser)
tr, va, te = particao(0)
Xz, mu, sd = padronizar(X, tr)
for nome, i in [("treino", tr), ("validação", va), ("teste", te)]:
    print(f"{nome:10s} média={Xz[i].mean():+.3f}  desvio={Xz[i].std():.3f}")

treino     média=-0.000  desvio=1.000
validação  média=+0.024  desvio=1.038
teste      média=-0.242  desvio=1.011


## 6. Salvar a base do trabalho

Um único arquivo com tudo que os outros notebooks precisam (~45 MB). No Colab, ele fica em `Meu Drive/trabalho_series_temporais/dados/`.

In [6]:
np.savez_compressed(
    DADOS / "minds_st.npz",
    X=X, cols=cols, mask=mask, y=y, classes=classes, arquivo=arquivo,
    sinalizante=sinalizante, repeticao=repeticao, grupo=grupo,
    validacao=np.array([VALIDACAO[k] for k in range(4)]),
    t0=idx.t0.to_numpy(), t1=idx.t1.to_numpy(), espelhado=idx.espelhado.to_numpy(),
)
divisao.to_csv(RES / "00_divisao_sinalizantes.csv", index=False)
print("salvo:", DADOS / "minds_st.npz", f"({(DADOS / 'minds_st.npz').stat().st_size / 1e6:.1f} MB)")

# conferência: recarregar e reconstruir a rodada 0
chk = np.load(DADOS / "minds_st.npz")
te0 = np.where(chk["grupo"] == 0)[0]
print("conferência:", chk["X"].shape, "| teste da rodada 0:", len(te0), "vídeos, sinalizantes", sorted(int(v) for v in set(chk["sinalizante"][te0])))

salvo: /home/claude/ts/dados/minds_st.npz (35.2 MB)
conferência: (1158, 32, 321) | teste da rodada 0: 275 vídeos, sinalizantes [1, 5, 9]


## Resumo

- **Base:** 1.158 séries multivariadas do MINDS-Libras, cada uma com 32 instantes × 321 atributos, 20 palavras, 12 sinalizantes.
- **Divisão:** 4 rodadas, cada uma testando 3 sinalizantes nunca vistos (7 no treino, 2 na validação). É o equivalente, para séries independentes, da divisão cronológica: nunca há sorteio, e o teste simula pessoas novas.
- **Sem vazamento:** z-score ajustado só no treino de cada rodada; pré-processamentos aplicados série a série.
- **Saída:** `dados/minds_st.npz`, lido por todos os notebooks seguintes.